In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Stage-I ybar inference runner (Noise2Noise -> ybar)
---------------------------------------------------
This script:
- Reads an Excel split sheet (columns like: 'batch', 'random_num', 'noise_file')
- Selects subset by fold and split (train / val / test / trainval)
- Runs a trained Unet2D (Noise2Noise) to predict ybar for each volume
- Saves outputs as:
    * <stem>_ybar.nii.gz              (model-space, for Stage-II training)
    * <stem>_ybar_restored.nii.gz     (display-space, for visual comparison; DEFAULT ON)

Key points:
- Correct neighbor stacking order -> [Z, 2, H, W]
- Safe CUDA FP16 inference (optional via --no-half to disable)
- torch.compile is OFF by default (no compiler needed); enable with --compile
- Overwrite is ON by default; disable with --no-overwrite
- Restored saving is ON by default; disable with --no-restored
- Jupyter-friendly: avoids -f argv issue by defaulting argv=[]
"""

import os
import sys
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nb
from tqdm import tqdm
import torch

# ---------- 0) Make your project importable ----------
# Change this path if your project root differs.
sys.path.append(r"/host/c/Users/ROG/Documents/Github")

from CTDenoising_Diffusion_N2N.Data_processing import (
    apply_transfer_to_img,
    cutoff_intensity,
    normalize_image,   # used in preprocess; inverse done manually for robustness
    crop_or_pad,
)

# ---------- 1) Robust import for Unet2D ----------
def import_unet2d():
    """
    Prefer your confirmed path; fallback to scanning package if import fails.
    """
    try:
        from CTDenoising_Diffusion_N2N.noise2noise.model import Unet2D
        return Unet2D
    except Exception:
        import pkgutil, importlib, CTDenoising_Diffusion_N2N as _pkg
        for m in pkgutil.walk_packages(_pkg.__path__, _pkg.__name__ + "."):
            try:
                mod = importlib.import_module(m.name)
                if hasattr(mod, "Unet2D"):
                    print(f"[info] Fallback import: Unet2D from {m.name}")
                    return getattr(mod, "Unet2D")
            except Exception:
                pass
        raise ImportError("Cannot import Unet2D. Please verify package path/filename.")

Unet2D = import_unet2d()

# ---------- 2) IO helpers ----------
def load_histogram_bins(bins_path: Path, bins_mapped_path: Path, enable: bool):
    """Load histogram equalization LUTs if enabled."""
    if not enable:
        return None, None
    if not bins_path.exists() or not bins_mapped_path.exists():
        raise FileNotFoundError(f"LUTs not found: {bins_path} or {bins_mapped_path}")
    bins = np.load(str(bins_path), allow_pickle=True)
    bins_mapped = np.load(str(bins_mapped_path), allow_pickle=True)
    return bins, bins_mapped

def preprocess_volume(vol: np.ndarray,
                      H: int, W: int,
                      hist_eq: bool,
                      bins, bins_mapped,
                      bg_cut: float, max_cut: float, norm: float) -> np.ndarray:
    """
    Apply *exactly the same* preprocessing as training:
      1) optional histogram equalization via LUTs
      2) intensity cutoff to [bg_cut, max_cut]
      3) normalization by `norm`
      4) crop/pad XY to fixed HxW (Z unchanged)
    Assumes vol: [H0, W0, Z] -> returns [H, W, Z].
    """
    if hist_eq:
        vol = apply_transfer_to_img(vol, bins, bins_mapped)
    vol = cutoff_intensity(vol, cutoff_low=bg_cut, cutoff_high=max_cut)
    vol = normalize_image(vol, normalize_factor=norm, image_max=max_cut, image_min=bg_cut, invert=False)
    vol = crop_or_pad(vol, [H, W, vol.shape[2]], value=float(vol.min()))
    return vol

def _invert_hist_eq(y: np.ndarray, bins, bins_mapped) -> np.ndarray:
    """
    Try to invert histogram mapping via project function; if it doesn't support reverse,
    fall back to a simple 1D inverse LUT with numpy.interp.
    """
    try:
        return apply_transfer_to_img(y, bins, bins_mapped, reverse=True)
    except TypeError:
        # Fallback: interpolate from mapped->original
        xv = np.asarray(bins_mapped).reshape(-1)
        yv = np.asarray(bins).reshape(-1)
        order = np.argsort(xv)
        xv, yv = xv[order], yv[order]
        y_flat = y.reshape(-1)
        y_inv = np.interp(y_flat, xv, yv).reshape(y.shape)
        return y_inv.astype(np.float32)

def inverse_to_display_space(ybar: np.ndarray,
                             H0: int, W0: int,
                             hist_eq: bool,
                             bins, bins_mapped,
                             bg_cut: float, max_cut: float, norm: float) -> np.ndarray:
    """
    Map model-space ybar back to display-space for visual comparison:
      1) inverse normalization: multiply by `norm`, clip to [bg_cut, max_cut]
      2) inverse histogram equalization if used
      3) crop/pad XY back to the original H0 x W0
    Returns: [H0, W0, Z], float32
    """
    y = (ybar.astype(np.float32) * float(norm))
    y = np.clip(y, bg_cut, max_cut)

    if hist_eq and (bins is not None) and (bins_mapped is not None):
        y = _invert_hist_eq(y, bins, bins_mapped)

    y = crop_or_pad(y, [H0, W0, y.shape[2]], value=float(y.min()))
    return y.astype(np.float32)

def save_both_spaces(ybar_model: np.ndarray,
                     nii_like: nb.Nifti1Image,
                     out_dir: Path,
                     stem: str,
                     save_restored: bool,
                     hist_eq: bool,
                     bins, bins_mapped,
                     bg_cut: float, max_cut: float, norm: float):
    """
    Save:
      - <stem>_ybar.nii.gz            (model-space; always saved)
      - <stem>_ybar_restored.nii.gz   (display-space; saved if save_restored=True)
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # model-space
    model_path = out_dir / f"{stem}_ybar.nii.gz"
    nb.save(nb.Nifti1Image(ybar_model.astype(np.float32), nii_like.affine, nii_like.header), str(model_path))

    disp_path = None
    if save_restored:
        H0, W0, _ = nii_like.shape
        y_disp = inverse_to_display_space(
            ybar=ybar_model, H0=H0, W0=W0,
            hist_eq=hist_eq, bins=bins, bins_mapped=bins_mapped,
            bg_cut=bg_cut, max_cut=max_cut, norm=norm
        )
        disp_path = out_dir / f"{stem}_ybar_restored.nii.gz"
        nb.save(nb.Nifti1Image(y_disp, nii_like.affine, nii_like.header), str(disp_path))

    return model_path, disp_path

# ---------- 3) inference ----------
@torch.no_grad()
def run_n2n_on_volume(model: torch.nn.Module,
                      vol_norm: np.ndarray,
                      batch_z: int = 32,
                      use_half: bool = True) -> np.ndarray:
    """
    Correct Z-batched inference.
    input  vol_norm: [H, W, Z]
    build  pairs   : [Z, 2, H, W]  (neighbors [z-1, z+1], edge-clamped)
    output ybar    : [H, W, Z] float32
    """
    device = next(model.parameters()).device
    H0, W0, Z = map(int, vol_norm.shape)

    z_idx = np.arange(Z)
    z0 = np.maximum(z_idx - 1, 0)
    z2 = np.minimum(z_idx + 1, Z - 1)

    a = np.ascontiguousarray(vol_norm[:, :, z0])    # [H,W,Z]
    b = np.ascontiguousarray(vol_norm[:, :, z2])    # [H,W,Z]

    pairs = np.stack([a, b], axis=0)                # [2,H,W,Z]
    pairs = np.transpose(pairs, (3, 0, 1, 2))       # [Z,2,H,W]
    pairs = np.ascontiguousarray(pairs)

    pairs_t = torch.from_numpy(pairs).float()

    if device.type == "cuda" and use_half:
        model = model.half()
        pairs_t = pairs_t.half()

    if device.type == "cuda":
        pairs_t = pairs_t.pin_memory()

    ybar = np.zeros((H0, W0, Z), dtype=np.float32)
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

    for i in range(0, Z, batch_z):
        j = min(i + batch_z, Z)
        batch = pairs_t[i:j].to(device, non_blocking=True)   # [Bz,2,H,W]
        if device.type == "cuda" and use_half:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                out = model(batch)                           # [Bz,1,H,W]
        else:
            out = model(batch)
        out_np = out.squeeze(1).detach().cpu().float().numpy()  # [Bz,H,W]
        ybar[:, :, i:j] = np.transpose(out_np, (1, 2, 0))

    if np.isnan(ybar).any() or np.isinf(ybar).any():
        ybar = np.nan_to_num(ybar, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return ybar

def load_n2n_model(ckpt_path: Path,
                   channels: int = 2,
                   out_dim: int = 1,
                   init_dim: int = 16,
                   dim_mults=(2, 4, 8, 16),
                   try_compile: bool = False) -> torch.nn.Module:
    """
    Instantiate Unet2D and load a checkpoint.
    Accepts:
      - raw state_dict
      - dict with {'model', optional 'ema'}
    Prefers EMA weights if available.
    """
    model = Unet2D(channels=channels, out_dim=out_dim, init_dim=init_dim, dim_mults=dim_mults)
    state = torch.load(str(ckpt_path), map_location="cpu")

    if isinstance(state, dict) and "model" in state:
        model.load_state_dict(state["model"], strict=False)
        if "ema" in state:
            try:
                from ema_pytorch import EMA
                ema = EMA(model)
                ema.load_state_dict(state["ema"])
                model = ema.ema_model
                print("[info] Loaded EMA weights.")
            except Exception:
                pass
    else:
        model.load_state_dict(state, strict=False)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    for p in model.parameters():
        p.requires_grad = False

    if try_compile:
        try:
            model = torch.compile(model)
            print("[info] torch.compile enabled.")
        except Exception as e:
            print(f"[warn] torch.compile failed ({e}); falling back to eager.")
    return model

def select_subset(df: pd.DataFrame,
                  fold: int,
                  use_split: str,
                  use_randoms=(0, 1)) -> pd.DataFrame:
    """
    Select subset by cross-validation fold and split.
    Convention:
      test fold = batches[fold]
      val  fold = batches[(fold + 1) % K]
      train     = all except (val + test)
    """
    if "batch" not in df.columns:
        raise RuntimeError("Excel must contain a 'batch' column.")

    batches = sorted(df["batch"].dropna().astype(int).unique().tolist())
    K = len(batches)
    if fold < 0 or fold >= K:
        raise ValueError(f"fold={fold} out of range. Available folds: {batches}")

    test_batch = batches[fold]
    val_batch  = batches[(fold + 1) % K]

    if use_split == "train":
        df_use = df[~df["batch"].astype(int).isin([test_batch, val_batch])]
    elif use_split == "val":
        df_use = df[df["batch"].astype(int) == val_batch]
    elif use_split == "test":
        df_use = df[df["batch"].astype(int) == test_batch]
    elif use_split == "trainval":
        df_use = df[~df["batch"].astype(int).isin([test_batch])]
    else:
        raise ValueError("use_split must be one of: train | val | test | trainval")

    if "random_num" in df_use.columns and use_randoms is not None:
        df_use = df_use[df_use["random_num"].isin(list(use_randoms))]
    return df_use

# ---------- 4) Main ----------
def main(argv=None):
    parser = argparse.ArgumentParser(
        description="Run Stage-I ybar inference with a trained N2N model.",
        allow_abbrev=False
    )
    # Paths
    parser.add_argument("--excel",   type=str, default=r"/host/d/file/fixedCT_static_simulation_train_test_gaussian_NAS.xlsx")
    parser.add_argument("--ckpt",    type=str, default=r"/host/d/file/noise2noise/model-78.pt")
    parser.add_argument("--outdir",  type=str, default=r"/host/d/file/outputs_ybar")
    # Split control
    parser.add_argument("--fold",    type=int, default=0)
    parser.add_argument("--split",   type=str, default="train", choices=["train", "val", "test", "trainval"])
    parser.add_argument("--randoms", type=int, nargs="*", default=[0, 1])
    # Preprocessing
    parser.add_argument("--H", type=int, default=256)
    parser.add_argument("--W", type=int, default=256)
    parser.add_argument("--hist-eq", action="store_true", help="enable histogram equalization (requires LUTs)")
    parser.add_argument("--bins",        type=str, default=r"/host/d/file/histogram_equalization/bins.npy")
    parser.add_argument("--bins-mapped", type=str, default=r"/host/d/file/histogram_equalization/bins_mapped.npy")
    parser.add_argument("--bg-cut", type=float, default=0.0)
    parser.add_argument("--max-cut", type=float, default=2000.0)
    parser.add_argument("--norm",   type=float, default=1000.0)
    # Inference speed knobs
    parser.add_argument("--batch-z",  type=int, default=32, help="slice-pairs per forward pass")
    parser.add_argument("--no-half",  action="store_true", help="disable FP16 on CUDA")
    parser.add_argument("--compile",  action="store_true", help="enable torch.compile (needs compiler toolchain)")
    # Overwrite & restored saving (defaults: overwrite=True, save_restored=True)
    ow = parser.add_mutually_exclusive_group()
    ow.add_argument("--overwrite",     dest="overwrite", action="store_true",  help="overwrite existing outputs (default)")
    ow.add_argument("--no-overwrite",  dest="overwrite", action="store_false", help="skip if output exists")
    parser.set_defaults(overwrite=True)

    rst = parser.add_mutually_exclusive_group()
    rst.add_argument("--save-restored", dest="save_restored", action="store_true",  help="save restored display-space ybar (default)")
    rst.add_argument("--no-restored",   dest="save_restored", action="store_false", help="do not save restored output")
    parser.set_defaults(save_restored=True)

    # In notebooks, kill Jupyter's '-f .../kernel.json'
    if argv is None and 'ipykernel' in sys.modules:
        argv = []
    args = parser.parse_args(argv)

    excel_path = Path(args.excel)
    ckpt_path  = Path(args.ckpt)
    out_dir    = Path(args.outdir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Env info
    cuda = torch.cuda.is_available()
    dev_name = torch.cuda.get_device_name(0) if cuda else "CPU"
    print(f"[env] CUDA: {cuda} | Device: {dev_name}")

    # Load table & subset
    df = pd.read_excel(str(excel_path))
    df_use = select_subset(df, fold=args.fold, use_split=args.split, use_randoms=args.randoms)
    if "noise_file" not in df_use.columns:
        raise RuntimeError("Excel must contain a 'noise_file' column.")

    paths = sorted(df_use["noise_file"].dropna().astype(str).unique().tolist())
    print(f"[split] fold={args.fold}, split={args.split}, volumes={len(paths)}")
    if {"batch", "random_num"}.issubset(df_use.columns):
        print("[dist] batch x random_num counts:")
        try:
            print(df_use.groupby(["batch", "random_num"]).size())
        except Exception:
            pass

    # Histogram LUTs
    bins, bins_mapped = load_histogram_bins(Path(args.bins), Path(args.bins_mapped), args.hist_eq)

    # Model
    model = load_n2n_model(
        ckpt_path=ckpt_path,
        channels=2, out_dim=1,
        init_dim=16, dim_mults=(2, 4, 8, 16),
        try_compile=args.compile
    )
    use_half = (not args.no_half) and (next(model.parameters()).device.type == "cuda")

    saved = failed = skipped = 0

    # Inference
    for pth in tqdm(paths, desc="ybar inference"):
        try:
            vol_path = Path(pth)
            stem = vol_path.name.replace(".nii.gz","").replace(".nii","")
            out_model = out_dir / f"{stem}_ybar.nii.gz"

            if out_model.exists() and not args.overwrite:
                skipped += 1
                continue

            nii = nb.load(str(vol_path))
            vol = nii.get_fdata().astype(np.float32)     # [H0, W0, Z]

            vol_n = preprocess_volume(vol,
                                      H=args.H, W=args.W,
                                      hist_eq=args.hist_eq,
                                      bins=bins, bins_mapped=bins_mapped,
                                      bg_cut=args.bg_cut, max_cut=args.max_cut, norm=args.norm)

            ybar = run_n2n_on_volume(model, vol_n, batch_z=args.batch_z, use_half=use_half)

            # save model-space + (default) display-space
            save_both_spaces(
                ybar_model=ybar,
                nii_like=nii,
                out_dir=out_dir,
                stem=stem,
                save_restored=args.save_restored,       # DEFAULT True now
                hist_eq=args.hist_eq, bins=bins, bins_mapped=bins_mapped,
                bg_cut=args.bg_cut, max_cut=args.max_cut, norm=args.norm
            )
            saved += 1

        except Exception as e:
            print(f"[skip] {pth} failed: {e}")
            failed += 1

    print(f"✅ Done. saved={saved}, failed={failed}, skipped={skipped} -> {out_dir}")

if __name__ == "__main__":
    main()

ImportError: cannot import name 'Unet2D' from 'CTDenoising_Diffusion_N2N.noise2noise' (unknown location)